# Advanced PTQ: ResNet-18

This notebook uses the experimental `advanced_ptq.py` module. It keeps the PTQ math visible for asymmetric activation quantization, MSE-based range search, INT32 accumulation, fixed-point requantization, and reference integer Dense/Conv2D operations.

This is a study/reference notebook. It does not replace TensorFlow Lite for full optimized integer ResNet-18 inference.

## Setup

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
import importlib

import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from tensorflow import keras

from src.models import load_pretrained_resnet18

advanced_ptq_module = importlib.import_module(
    "src.quantization.custom_quantization.advanced_ptq"
)
advanced_ptq_module = importlib.reload(advanced_ptq_module)

asymmetric_quantization_params = advanced_ptq_module.asymmetric_quantization_params
calibrate_activation_ranges_mse = advanced_ptq_module.calibrate_activation_ranges_mse
dequantize_advanced_tensor = advanced_ptq_module.dequantize_tensor
integer_conv2d_nhwc = advanced_ptq_module.integer_conv2d_nhwc
integer_dense = advanced_ptq_module.integer_dense
mse_optimal_asymmetric_range = advanced_ptq_module.mse_optimal_asymmetric_range
mse_optimal_symmetric_range = advanced_ptq_module.mse_optimal_symmetric_range
quantize_tensor = advanced_ptq_module.quantize_tensor
requantize_int32 = advanced_ptq_module.requantize_int32
symmetric_quantization_params = advanced_ptq_module.symmetric_quantization_params

## Load CIFAR-10 Dataset

CIFAR-10 images are used as representative inputs for calibration and metric checks. The pretrained ResNet-18 model is ImageNet-trained, so CIFAR-10 labels are metadata only.

In [ ]:
(cifar_train_images, cifar_train_labels), _ = keras.datasets.cifar10.load_data()

num_cifar_samples = 100
raw_images = [
    np.asarray(
        Image.fromarray(image).resize((224, 224), Image.Resampling.BILINEAR),
        dtype=np.uint8,
    )[None, ...]
    for image in cifar_train_images[:num_cifar_samples]
]
cifar_labels = cifar_train_labels[:num_cifar_samples].reshape(-1)

len(raw_images), raw_images[0].shape, raw_images[0].dtype, cifar_labels[:10]

## Load Model And Preprocess Samples

In [ ]:
model = load_pretrained_resnet18()
samples = [np.asarray(model.preprocessor(image), dtype=np.float32) for image in raw_images]

samples[0].shape, samples[0].dtype

## MSE-Based Activation Calibration

This step searches activation ranges that minimize reconstruction error after quantize/dequantize. It uses asymmetric activation quantization with a zero-point.

In [ ]:
num_calibration_samples = 25
num_bits = 8
activation_dtype = "uint8"
num_mse_candidates = 20

activation_params = calibrate_activation_ranges_mse(
    model,
    samples[:num_calibration_samples],
    num_bits=num_bits,
    dtype=activation_dtype,
    num_candidates=num_mse_candidates,
)

activation_table = pd.DataFrame([
    {
        "layer": layer_name,
        "scale": float(params.scale),
        "zero_point": int(params.zero_point),
        "qmin": params.qmin,
        "qmax": params.qmax,
        "num_bits": params.num_bits,
        "dtype": params.dtype,
        "symmetric": params.symmetric,
    }
    for layer_name, params in activation_params.items()
])

activation_table

## Inspect the Quantized Final Output

This checks how much quantize/dequantize changes the final output tensor for one sample.

In [ ]:
fp32_output = model(samples[0], training=False).numpy()

output_min, output_max, output_mse = mse_optimal_asymmetric_range(
    fp32_output,
    num_bits=num_bits,
    dtype=activation_dtype,
    num_candidates=50,
)
output_params = asymmetric_quantization_params(
    output_min,
    output_max,
    num_bits=num_bits,
    dtype=activation_dtype,
)
quantized_output = quantize_tensor(fp32_output, output_params)
reconstructed_output = dequantize_advanced_tensor(quantized_output)

fp32_top_class = int(np.argmax(fp32_output[0]))
quantized_top_class = int(np.argmax(quantized_output.values[0]))

pd.DataFrame([
    {
        "tensor": "final_output",
        "dataset": "cifar10",
        "cifar10_label": int(cifar_labels[0]),
        "fp32_top_class_index": fp32_top_class,
        "quantized_top_class_index": quantized_top_class,
        "matches_fp32_prediction": quantized_top_class == fp32_top_class,
        "max_abs_output_difference": float(np.max(np.abs(fp32_output - reconstructed_output))),
        "mean_abs_output_difference": float(np.mean(np.abs(fp32_output - reconstructed_output))),
        "mse_range_search_error": output_mse,
        "output_scale": float(output_params.scale),
        "output_zero_point": int(output_params.zero_point),
    }
])

## Reference Integer Dense Layer

This cell finds the final Dense layer, quantizes its input activations asymmetrically, quantizes weights symmetrically, accumulates with INT32, and requantizes the output.

In [ ]:
def iter_all_layers(layer):
    if hasattr(layer, "layers"):
        for child in layer.layers:
            yield from iter_all_layers(child)
    yield layer

all_layers = list(iter_all_layers(model))
dense_layers = [layer for layer in all_layers if isinstance(layer, keras.layers.Dense)]
classifier_dense = dense_layers[-1]

classifier_input_model = keras.Model(
    inputs=model.input,
    outputs=classifier_dense.input,
)
classifier_input = classifier_input_model(samples[0], training=False).numpy()
classifier_weights = classifier_dense.get_weights()
classifier_kernel = classifier_weights[0]
classifier_bias = classifier_weights[1] if len(classifier_weights) > 1 else None

fp32_dense_output = classifier_input @ classifier_kernel
if classifier_bias is not None:
    fp32_dense_output = fp32_dense_output + classifier_bias

integer_dense_result = integer_dense(
    classifier_input,
    classifier_kernel,
    classifier_bias,
)

pd.DataFrame([
    {
        "layer": classifier_dense.name,
        "input_shape": tuple(classifier_input.shape),
        "kernel_shape": tuple(classifier_kernel.shape),
        "accumulator_dtype": str(integer_dense_result.accumulator_int32.dtype),
        "output_int_dtype": str(integer_dense_result.output_int.dtype),
        "fp32_top_class_index": int(np.argmax(fp32_dense_output[0])),
        "integer_dequant_top_class_index": int(np.argmax(integer_dense_result.output_dequantized[0])),
        "matches_fp32_dense_prediction": int(np.argmax(fp32_dense_output[0])) == int(np.argmax(integer_dense_result.output_dequantized[0])),
        "max_abs_dense_difference": float(np.max(np.abs(fp32_dense_output - integer_dense_result.output_dequantized))),
        "mean_abs_dense_difference": float(np.mean(np.abs(fp32_dense_output - integer_dense_result.output_dequantized))),
        "input_scale": float(integer_dense_result.input_params.scale),
        "weight_scale": float(integer_dense_result.weight_params.scale),
        "output_scale": float(integer_dense_result.output_params.scale),
        "output_zero_point": int(integer_dense_result.output_params.zero_point),
    }
])

## Reference Integer Conv2D Layer

This demonstrates INT32 accumulation on the first Conv2D kernel using a small real preprocessed image patch. It is a reference check for the math, not a full ResNet forward pass.

In [ ]:
conv_layers = [
    layer
    for layer in all_layers
    if isinstance(layer, keras.layers.Conv2D) and layer.get_weights()
]
first_conv = conv_layers[0]
conv_weights = first_conv.get_weights()
conv_kernel = conv_weights[0]
conv_bias = conv_weights[1] if len(conv_weights) > 1 else None
kernel_h, kernel_w, in_channels, _ = conv_kernel.shape

conv_patch = samples[0][:, : kernel_h + 4, : kernel_w + 4, :in_channels]
stride = first_conv.strides
padding = first_conv.padding

fp32_conv_output = tf.nn.conv2d(
    conv_patch,
    conv_kernel,
    strides=(1, stride[0], stride[1], 1),
    padding=padding.upper(),
).numpy()
if conv_bias is not None:
    fp32_conv_output = fp32_conv_output + conv_bias.reshape((1, 1, 1, -1))

integer_conv_result = integer_conv2d_nhwc(
    conv_patch,
    conv_kernel,
    conv_bias,
    strides=stride,
    padding=padding,
)

pd.DataFrame([
    {
        "layer": first_conv.name,
        "patch_shape": tuple(conv_patch.shape),
        "kernel_shape": tuple(conv_kernel.shape),
        "strides": stride,
        "padding": padding,
        "accumulator_dtype": str(integer_conv_result.accumulator_int32.dtype),
        "output_int_dtype": str(integer_conv_result.output_int.dtype),
        "fp32_output_shape": tuple(fp32_conv_output.shape),
        "integer_output_shape": tuple(integer_conv_result.output_dequantized.shape),
        "max_abs_conv_difference": float(np.max(np.abs(fp32_conv_output - integer_conv_result.output_dequantized))),
        "mean_abs_conv_difference": float(np.mean(np.abs(fp32_conv_output - integer_conv_result.output_dequantized))),
        "input_scale": float(integer_conv_result.input_params.scale),
        "weight_scale": float(integer_conv_result.weight_params.scale),
        "output_scale": float(integer_conv_result.output_params.scale),
        "output_zero_point": int(integer_conv_result.output_params.zero_point),
    }
])

## Range Search Comparison

This compares simple min/max scaling with MSE-selected clipping for the classifier kernel.

In [ ]:
weight_max_abs = float(np.max(np.abs(classifier_kernel)))
minmax_weight_params = symmetric_quantization_params(weight_max_abs, num_bits=num_bits)
minmax_weight_quantized = quantize_tensor(classifier_kernel, minmax_weight_params)
minmax_weight_reconstructed = dequantize_advanced_tensor(minmax_weight_quantized)
minmax_weight_mse = float(np.mean((classifier_kernel - minmax_weight_reconstructed) ** 2))

mse_clip_abs, mse_weight_error = mse_optimal_symmetric_range(
    classifier_kernel,
    num_bits=num_bits,
    num_candidates=100,
)
mse_weight_params = symmetric_quantization_params(mse_clip_abs, num_bits=num_bits)
mse_weight_quantized = quantize_tensor(
    np.clip(classifier_kernel, -mse_clip_abs, mse_clip_abs), mse_weight_params
)
mse_weight_reconstructed = dequantize_advanced_tensor(mse_weight_quantized)

pd.DataFrame([
    {
        "method": "minmax_symmetric",
        "clip_abs": weight_max_abs,
        "scale": float(minmax_weight_params.scale),
        "zero_point": int(minmax_weight_params.zero_point),
        "reconstruction_mse": minmax_weight_mse,
    },
    {
        "method": "mse_symmetric",
        "clip_abs": mse_clip_abs,
        "scale": float(mse_weight_params.scale),
        "zero_point": int(mse_weight_params.zero_point),
        "reconstruction_mse": mse_weight_error,
    },
])

## Save Summary Results

In [ ]:
output_dir = PROJECT_ROOT / "artifacts" / "resnet18_advanced_ptq_notebook"
output_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "dataset": "cifar10",
    "num_input_samples": len(samples),
    "num_calibration_samples": num_calibration_samples,
    "num_bits": num_bits,
    "activation_dtype": activation_dtype,
    "num_activation_layers_calibrated": len(activation_params),
    "final_output_quantized": {
        "fp32_top_class_index": fp32_top_class,
        "quantized_top_class_index": quantized_top_class,
        "matches_fp32_prediction": quantized_top_class == fp32_top_class,
        "max_abs_output_difference": float(np.max(np.abs(fp32_output - reconstructed_output))),
        "mean_abs_output_difference": float(np.mean(np.abs(fp32_output - reconstructed_output))),
        "scale": float(output_params.scale),
        "zero_point": int(output_params.zero_point),
    },
    "integer_dense": {
        "layer": classifier_dense.name,
        "input_shape": tuple(classifier_input.shape),
        "kernel_shape": tuple(classifier_kernel.shape),
        "max_abs_difference": float(np.max(np.abs(fp32_dense_output - integer_dense_result.output_dequantized))),
        "mean_abs_difference": float(np.mean(np.abs(fp32_dense_output - integer_dense_result.output_dequantized))),
    },
    "integer_conv2d": {
        "layer": first_conv.name,
        "patch_shape": tuple(conv_patch.shape),
        "kernel_shape": tuple(conv_kernel.shape),
        "max_abs_difference": float(np.max(np.abs(fp32_conv_output - integer_conv_result.output_dequantized))),
        "mean_abs_difference": float(np.mean(np.abs(fp32_conv_output - integer_conv_result.output_dequantized))),
    },
}

summary_path = output_dir / "results.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
summary_path

## Notes

- This notebook demonstrates advanced PTQ pieces separately.
- `calibrate_activation_ranges_mse` computes asymmetric activation scales and zero-points from representative samples.
- `integer_dense` and `integer_conv2d_nhwc` use INT32 accumulation and then requantize outputs.
- The notebook does not perform a full integer ResNet-18 graph rewrite. TensorFlow Lite remains the built-in full-model integer inference reference.